# Analyse des résultats du Baromètre Science Ouverte de Paris 8

## Le BSO national
Le Baromètre Science Ouverte (BSO) donne à voir l'évolution du nombre de publications dont l'accès est dit « ouvert ». Réalisé la première fois  en 2018, il est mis à jour tous les ans. Par "accès ouvert", le BSO désigne "les publications de recherche mises à disposition librement sur l'internet public" (Le glossaire du BSO [https://barometredelascienceouverte.esr.gouv.fr/a-propos/glossaire](https://barometredelascienceouverte.esr.gouv.fr/a-propos/glossaire)). Prenons deux articles :

1. Stark P., Adil F., Woodward B., Treviño J., Steinhoff C., Gallegos D., 2024, « Real-time Awareness: A Novel Application of Fiber Optics to Optimize Simultaneous Operations of Drilling and Frac », *SPE Hydraulic Fracturing Technology Conference and Exhibition*, p. D031S008R001. https://doi.org/10.2118/217777-MS

2. Firooz S., Reddy B.D., Steinmann P., 2025, « A gradient-enhanced approach for stable finite element approximations of reaction-convection-diffusion problems », *Journal of Theoretical, Computational and Applied Mechanics*, p. 15788. https://doi.org/10.46298/jtcam.15788

Le premier article est dit **"fermé"**, car il est nécessaire d'être abonné à la revue ou d'acheter l'article pour le lire. *A contrario*, le second, que nous avons trouvé sur le Web, est **"ouvert"** : on peut y accèder sans autre condition qu'un accès à Internet.

Selon les observations faites en 2024 par le ministère de l'Enseignement supérieur, de la Recherche et de l'Innovation (MESRI), 105 224 des 160681 publications parues en 2023 et disposant d'un "DOI Crossref" étaient en "accès ouvert", soit un **taux d'ouverture de 66%**. 


## Les baromètres science ouverte locaux

En 2019, l'université de Lorraine reprend la méthode utilisée par le MESRI pour réaliser une déclinaison locale du BSO. Depuis, d'autres établissements d'enseignement supérieur ont voulu avoir leur propre baromètre, conduisant le ministère a élaboré en 2022 une procédure simplifiée. Il "suffit" en effet de télécharger sur le [site du BSO](https://barometredelascienceouverte.esr.gouv.fr/declinaisons/bso-locaux) un fichier CSV contenant les doi des publications de l'établissement, son identifiant ou le nom de sa collection Hal. Ensuite les équipes du BSO extraient depuis leur base de données les publications de l'établissement et leur métadonnées associées. Une fois l'extraction terminée, les résultats peuvent être visualisés directement sur la page du BSO ou depuis une page web de l'établissement.




In [ ]:
#chargement des modules
import pandas as pd
import datetime
import os
import json
from io import StringIO
from myst_nb import glue
import numpy as np
import plotly.express as px

In [ ]:
import s3fs #pour connecter au bucket

## Le baromètre de Paris 8

Au printemps 2025, la bibliothèque de Paris 8 a donc décidé de plonger dans Hal et Pubmed pour réaliser son propre baromètre science ouverte local. Les visualisations du baromètre local de Paris 8 sont maintenat disponibles sur le [site web de la bibliothèque](https://www.bu.univ-paris8.fr/services/services-aux-chercheurs/le-barometre-science-ouverte-a-paris-8/). On peut également y accéder via le site du BSO national : Il suffit de se rendre à l'adresse  [https://barometredelascienceouverte.esr.gouv.fr/declinaisons/comment-realiser-bso-local](https://barometredelascienceouverte.esr.gouv.fr/declinaisons/comment-realiser-bso-local) et de rentrer le numéro **199318270** dans la rubrique "Identifiant de l'établissement". Les résultats porte sur la période 2016-2024. Une nouvelle demande a par ailleurs été faite pour intégrer les publications de 2025.

### Processus de collecte

Nous avons repris la procédure proposée par l'université de Lorraine en suivant les "notebooks" disponibles sur Gitlab. Nous avons également construit de nouveaux "notebooks" destiné à automatiser la collecte des données depuis les API de Hal et de Pubmed. Les notebooks sont disponibles sur le dépôt Framagit du SCD de Paris 8 : [collecte_hal](https://framagit.org/scd_paris8/barometre-sciences-ouvertes-universite-paris-8/-/blob/85f8bf985cfe07152056203f48e4dbeeeb638cca/0_script_collect/collecte_hal.ipynb) et [collecte_Pubmed](https://framagit.org/scd_paris8/barometre-sciences-ouvertes-universite-paris-8/-/blob/85f8bf985cfe07152056203f48e4dbeeeb638cca/0_script_collect/collecte_pubmid.ipynb). 

Concrètement, la première étape consiste à définir le périmètre du baromètre local en transmettant à l'équipe du BSO la liste des DOI trouvés sur Pubmed et Hal qui correspondent à des publications dont au moins un des auteurs est affilié à l'université Paris 8. On indique également l'identifiant Hal de l'établissement. Le fichier listant les identifiants (doi ou identifiant Hal) des publications associées à Paris 8 sur la période 2016-2024 contient 9029 lignes. Il est disponible sur [Data Paris 8](https://entrepot.recherche.data.gouv.fr/dataverse/univ-paris8) : 

> Luneau, Aymeric; Arneau, Stéphanie, 2026, "Fichiers envoyés au BSO pour établissement du baromètre science ouverte de Paris 8", [https://doi.org/10.57745/OMEMKY](https://doi.org/10.57745/OMEMKY), Recherche Data Gouv

L'équipe du BSO se charge ensuite de récupérer les métadonnées à partir de Crossref et de Hal. Outre les métadonnées rendues disponibles par les API, l'équipe du BSO a mis en place un ensemble de méthodes pour les enrichir : le statut d'ouverture, la catégorisations disciplinaires, identification de la langue, détecter le pays d'affiliation ([voir la méthodologie du BSO](https://barometredelascienceouverte.esr.gouv.fr/a-propos/methodologie)). Une fois le travail de collecte et de traitement, le BSO nous renvoie les jeu de données en même temps qu'il met en ligne les résultats du baromètre local. Ce jeu de données est constitué de deux fichiers :

En retour, les équipes du BSO renvoient deux fichiers de données : 

* un fichier ".csv" qui ne contient que les données observées lors de l'année en cours. Dans notre cas, le fichier "bso-publications-latest_199318270_enriched.csv" ne contient que le statut des publications observées en 2024.

* un fichier ".jsonl" (json lines) qui contient toutes les données en historique (statut OA pour chaque DOI et à chaque date d'observation).

En effet si un article scientifique n'a qu'une seule **"date de publication"** (date à laquelle il est publié). C’est une donnée qui ne varie pas au cours du temps. Le Baromètre mesure actuellement le taux d’ouverture des publications qui ont été publiées sur la période de 2013 à 2023, un décalage d’un an sur la dernière année de publication étant toujours observé dans le Baromètre national pour prendre en compte les embargos d’ouverture des publications. 

Par contre il peut avoir plusieurs **"date d'observation"**. En effet, depuis le premier baromètre en 2018, l'équipe du BSO produit chaque année une "photographie" de l'état d'ouverture des publications. L’état d’ouverture est ainsi conservé pour chaque année et permet de mesurer l’évolution du taux d’ouverture des publications. Ainsi, si un laboratoire veut comparer le taux d’ouverture de ses publications scientifiques entre deux périodes, il peut visualiser son taux d’ouverture en 2021 par rapport à celui de 2023 par exemple. 

> Claire de Cooman, Laetitia Bracco. Comment valoriser l'ouverture de la production scientifique d'un laboratoire ? Rapport sur l'implémentation d'indicateurs du Baromètre de la Science Ouverte sur deux pages de collection HAL. Université de Lorraine. 2024. ⟨hal-04708155⟩


### Exploiter la richesse des données du Baromètre local

Les analyses réalisées par l'équipe du BSO et rendues visibles par les graphiques publiés sur le site de la bibliothèque se concentrent sur la question de l'ouverture des publications. Elles donnent ainsi à voir l'évolution des voies (ou modalites) d'ouverture au fil des années ou le taux d'ouverture par champs disciplinaires (SHS, sciences biologiques, sciences physiques, etc.). La compréhension des résultats suppose toutefois de plonger dans les données pour saisir les différences entre les modalités d'ouverture des publications ({term}`voie diamant`, {term}`voie dorée`, {term}`voie verte`) ou expliquer pourquoi on a des publications en biologie ou en physique associées à Paris 8 (on verra plus loin que c'est lié à la classification automatique opérée par l'équipe du BSO).

Plonger dans les données permet surtout d'exploiter la richesse des informations qu'elles contiennent. Nous nous sommes en particulier intéressé aux liens tissés par Paris 8 à travers ces publications : les liens entre chercheurs, mais aussi les liens entre les organismes de recherche.

## Reconstruire le réseaux des institutions associées à Paris 8

Parmi les informations présentes dans les données du BSO local, il y en a qui portent sur les affiliations des chercheurs. Cela permet ainsi d'envisager une exploration des relations de Paris 8 avec les autres universités françaises ou étrangères. Toutefois, le travail des donnée a fait apparaître deux cas de figure :

1. les organismes mentionnés sont associés à leur numéro ROR.
2. il n'y a pas de ROR, mais on dispose d'une adresse postale.

Le deuxième cas de figure est plus compliqué puisqu'il suppose d'extraire des informations (nom d'organisme, nom de villes, de Pays) à partir d'information non structurées. Nous nous sommes donc concentrés sur les cas pour lesquels nous disposons du ROR. À cela s'ajoute le fait qu'il n'y a pas toujours d'information concernant les affiliations. Les résultats présentés ci-dessous sont donc partiels et sont à interpréter avec précaution.

### Récupération des informations grâce au ROR

Une fois les rors récupérés, nous avons interrogé l'API du Research Organisation Registry ([https://api.dev.ror.org/organizations](https://api.dev.ror.org/organizations)) puis extrait les données suivantes : le nom de l'organisme, lse noms de la ville, du pays et du continent, ainsi que la latitude et la longitude.




## Chargement des données via le bucket du ssp cloud

In [ ]:
# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"

In [ ]:
file_path = BUCKET_OUT + "/" + "Data_bso/fichier_from_mesr/bso-publications-latest_199318270_enriched.jsonl"
print(file_path)
list_id_pub = []
datas =[]
with fs.open(file_path, 'r', encoding='utf-8') as file:
    for line in file:
        try:
            data = json.loads(line)
            # Extract specific fields
            ids = data.get('all_ids')
            list_id_pub.append(ids[0])
            datas.append(data)
            
        except json.JSONDecodeError:
            continue  # Skip invalid lines


## Extraction des variables d'intérêt

In [ ]:
has_doi = []
has_halid = []
hal_id = []
is_oa = []
journal_is_oa = []
snapshot_date = []
observation_date = []
oa_host_type = []
oa_colors = []
unpaywall_oa_status = []
oa_colors_with_priority_to_publisher = []
repositories_concat = []
is_oa_hal = []
is_green = []
#published_date = []
pub_year = []
list_ids = []
bso_class = []
keywords = []
pub_type = []

for n, x in enumerate(list_id_pub):
    dict_pub = datas[n]
    list_keyword = []
    for y in dict_pub['oa_details']:
        dict_oa = dict_pub['oa_details'][y]
        list_ids.append(x)
        #published_date.append(dict_pub['external_ids']['published_date'])
        pub_year.append(str(dict_pub['year']))
        is_oa.append(dict_oa["is_oa"])
        if 'hal_id' in dict_pub:
            hal_id.append(dict_pub['hal_id'])
            has_halid.append(True)
        else:
            hal_id.append(np.nan)
            has_halid.append(False)
        if 'journal_is_oa' in dict_oa:
             journal_is_oa.append(dict_oa['journal_is_oa'])
        else:
             journal_is_oa.append(False)
        if 'doi' in dict_pub:
             has_doi.append(True)
        else:
             has_doi.append(False)

        bso_class.append(dict_pub['bso_classification'])
        if 'keywords' in dict_pub:
            for kw in dict_pub["keywords"]:
                list_keyword.append(kw["keyword"])
        else:
            list_keyword.append('no keywords')
        pub_type.append(dict_pub['genre'])
        
        snapshot_date.append(dict_oa["snapshot_date"])
        observation_date.append(dict_oa["observation_date"])
        oa_host_type.append(dict_oa["oa_host_type"])
        if 'unpaywall_oa_status' in dict_oa:
            unpaywall_oa_status.append(dict_oa['unpaywall_oa_status'])
        else:
            unpaywall_oa_status.append(np.nan)
        oa_colors.append(dict_oa["oa_colors"])
        oa_colors_with_priority_to_publisher.append(dict_oa["oa_colors_with_priority_to_publisher"][0])
        repositories_concat.append(dict_oa["repositories_concat"])
        is_oa_hal.append(dict_oa["is_oa_hal"])
        if 'green' in dict_oa["oa_colors"]:
            is_green.append(True)
        else:
            is_green.append(False)
        if len(dict_oa["oa_colors_with_priority_to_publisher"]) > 1:
            print(dict_oa["oa_colors_with_priority_to_publisher"])
            



In [ ]:
data_columns = {
    "id": list_ids,
    "pub_year" : pub_year,
    "has_doi" : has_doi,
    "has_halid": has_halid,
    "hal_id" : hal_id,
    "is_oa":is_oa,
    "journal_is_oa":journal_is_oa,
    "snapshot_date":snapshot_date,
    "observation_date":observation_date,
    "oa_host_type":oa_host_type,
    "oa_colors":oa_colors,
    "unpaywall_oa_status": unpaywall_oa_status,
    "oa_colors_with_priority_to_publisher":oa_colors_with_priority_to_publisher,
    "repositories_concat":repositories_concat,
    "is_green" : is_green,
    "is_oa_hal":is_oa_hal,
    "bso_class": bso_class,
    #"keywords": keywords,
    "publication_type": pub_type
}

df0 = pd.DataFrame(data= data_columns)


In [ ]:

df0['date'] = pd.to_datetime(pd.to_datetime(df0['pub_year']).dt.date) #nc convertit l'année au format date
df0['year_d'] = df0['date'].dt.year

df0['observation_year'] = df0.observation_date.str.replace('Q*','')
df0['observation_date_rec'] = pd.to_datetime(pd.to_datetime(df0['observation_year']).dt.date) #nc convertit l'année au format date
df0['observation_year_rec'] = df0['observation_date_rec'].dt.year




In [ ]:
df0.loc[df0.bso_class.str.contains("Biology \(fond")]

In [ ]:
df_complete = df0.loc[(df0.observation_year_rec == datetime.date.today().year-2) & ((df0.year_d<datetime.date.today().year-1)&(df0.year_d>2012))]



# Recodage des statuts manquants sur unpaywall (articles non référencés?
df_complete.loc[(df_complete.unpaywall_oa_status.isna()) & (df_complete.is_oa == False) & (df_complete.has_doi==True), "unpaywall_oa_status_rec"] = "closed"
df_complete.loc[(df_complete.unpaywall_oa_status.isna()) & (df_complete.is_oa == False) & (df_complete.has_doi==False), "unpaywall_oa_status_rec"] = "closed-unreferenced"
df_complete.loc[(df_complete.unpaywall_oa_status.isna()) & (df_complete.is_oa == True) & (df_complete.has_doi==False), "unpaywall_oa_status_rec"] = "opened-unreferenced"
df_complete.loc[(df_complete.unpaywall_oa_status.isna()) & (df_complete.is_oa == True) & (df_complete.has_doi==True), "unpaywall_oa_status_rec"] = "undefined"
df_complete.loc[~(df_complete.unpaywall_oa_status.isna()), "unpaywall_oa_status_rec"] = df_complete.unpaywall_oa_status


In [ ]:

oa_2023 = len(df_complete.loc[(df_complete.has_doi==True) & (df_complete.is_oa ==True) &  (df_complete.year_d==2023)])
pub_2023 = len(df_complete.loc[(df_complete.has_doi==True) & (df_complete.year_d==2023)])


taux_oa_2023 =  oa_2023/pub_2023*100
glue("taux_oa_2023", round(taux_oa_2023, ndigits =2))

oa_doi = len(df_complete.loc[(df_complete.has_doi==True) & (df_complete.is_oa ==True)])
pub_doi = len(df_complete.loc[(df_complete.has_doi==True)])


taux_oa_doi = oa_doi/pub_doi*100
glue("taux_oa_doi", round(taux_oa_doi, ndigits =2))


oa = len(df_complete.loc[(df_complete.is_oa ==True)])
pub = len(df_complete)

taux_oa = oa/pub*100
glue("taux_oa", round(taux_oa, ndigits =2))

data = {"Corpus":["Publications de 2023 avec doi", "Publications avec doi", "Toutes les publications"],
       "Accès ouvert": [oa_2023, oa_doi, oa],
       "Total": [pub_2023, pub_doi, pub],
       "Taux d'ouverture": [taux_oa_2023, taux_oa_doi, taux_oa]}

tab_taux_oa = pd.DataFrame(data)



In [ ]:
df_complete.unpaywall_oa_status_rec.unique()

In [ ]:
df_2023 = df_complete.loc[df_complete.year_d==2023]
oa_status = df_complete.groupby(["unpaywall_oa_status_rec","is_oa", "has_doi"]).agg(nb = ("id", "size")).sort_values("nb", ascending=False).reset_index()

oa_status1 = pd.concat([oa_status, oa_status[["nb"]].agg(["sum"])])
oa_status1.loc[oa_status1.unpaywall_oa_status_rec.isna(), "unpaywall_oa_status_rec"] = "Total"


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns # librairie pour visualiser les résultats
import squarify

## Les résultats du baromètre de Paris 8

En 2024 (année d'observation), le nombre de publication toutes années confondues est de 45485 et 13791 étaient en "accès ouvert". Si on ne prend en compte que les publications disposant d'un doi, le nombre total de publications et le nombre de publications en "accès ouvert" sont respectivement de {glue:text}`pub_doi` et {glue:text}`oa_doi`, soit un taux d'ouverture de {glue:text}`taux_oa_doi`%. Le taux d'ouverture des publications parues en 2023 est quant à lui de {glue:text}`taux_oa_2023`%, similaire aux taux d'ouverture observée à l'échelle nationale. 




In [ ]:
glue("tableau_oa", tab_taux_oa)

**Petite précision de vocabulaire**

Le fichier créé par le BSO contient tout type de publication : article, ouvrage, chapite d'ouvragre, résumé de conférence. Or, selon les champs disciplinaires, ces publications n'ont pas la même valeure. Par exemple, un papier soumis dans le cadre d'une conférence majeure en sciences informatiques peut être compté au même titre qu'un article publié dans une revue, tandis qu'en sociologie il est rare que les communications fassent l'objet de publication via des actes de colloques par exemple. Il peut donc paraître plus pertinent d'évaluer le taux d'ouverture en fonction du type de publication.

In [ ]:
d = df_complete.loc[df_complete.observation_year_rec==2024].groupby(["is_oa","publication_type"]).agg(nb=("id","size")).reset_index().fillna(0)

d_pivot = d.pivot(index = "publication_type", columns = "is_oa", values = "nb").fillna(0).reset_index().rename(columns={True:"open", False:"close"})

array_oa_pub = []

for p in d_pivot.publication_type.unique():
    print(p)
    nb_ouvert = [x for x in d_pivot["open"].loc[(d_pivot.publication_type==p)]]
    nb_closed = [x for x in d_pivot["close"].loc[(d_pivot.publication_type==p)]]

    total_pub = nb_ouvert[0]+nb_closed[0]
    array_oa_pub.append([nb_ouvert[0], nb_closed[0], total_pub])



In [ ]:
d.nb.loc[(d.publication_type=="book") & (d.is_oa==True)].reset_index().iloc[0]

In [ ]:
fig, axes = plt.subplots(2, figsize=(10,10))
df_grouped = df_complete.groupby(["year_d","has_doi"]).agg(nb =("id", "size")).sort_values("year_d").reset_index()

sns.barplot(data=df_grouped, x="year_d", y="nb", ax =axes[0], hue = "has_doi", dodge=False);

axes[0].set_xlabel("Années")
axes[0].set_ylabel("Nombre de publications")
axes[0].set_title("Nombre total de publications par an observé en 2024");


freq_doi = df_complete.groupby(["year_d", "has_doi"]).agg(nb =("id", "size")).sort_values("year_d").reset_index()
freq_total = df_complete.groupby(["year_d"]).agg(total_by_year =("id", "size")).sort_values("year_d").reset_index()
freq_doi = freq_doi.merge(freq_total, on = ["year_d"], how ="left")
freq_doi["prop"] = freq_doi["nb"]/freq_doi["total_by_year"]



 
# Data
x=[x for x in freq_doi.year_d.unique()]
y=[]
for n, z in enumerate(freq_doi.has_doi.unique()):
    v = [v for v in freq_doi.prop.loc[freq_doi.has_doi == z]]
    y.append(v)

# Plot
axes[1].stackplot(x,y, labels=['Sans DOI','Avec DOI'])
axes[1].set_xlabel("Années")
axes[1].set_ylabel("Proportion")
axes[1].legend(loc='upper left')
axes[1].set_title("Evolution du taux de publications avec un DOI crossref observées en 2024")
plt.tight_layout()

plt.show()

In [ ]:
import plotly
from IPython.core.display import HTML

In [ ]:
#fig, ax = plt.subplots(1, figsize=(10,10))
df_complete.loc[(df_complete.has_doi==True) & (df_complete.has_halid==False),"type_id"] = "doi sans hal_id"
df_complete.loc[(df_complete.has_doi==False) & (df_complete.has_halid==True),"type_id"] = "hal_id sans doi"
df_complete.loc[(df_complete.has_doi==True) & (df_complete.has_halid==True),"type_id"] = "doi avec hal_id"
df_complete.loc[(df_complete.has_doi==False) & (df_complete.has_halid==False),"type_id"] = "Sans identifiant"

df_grouped = df_complete.groupby(["year_d","type_id","has_doi"]).agg(nb =("id", "size")).sort_values(["year_d", "type_id"], ascending=[True, False]).reset_index()

fig = px.bar(df_grouped, x="year_d", y="nb", color = "type_id", title= "Nombre total de publications par an observé en 2024",
             labels = {"year_d":"Années de publication", "nb": "Nombre de publications"},)

config = {'displayModeBar':True}
fig.show()
#plotly.offline.plot(fig, filename = './figures/figure_1.html', config = config)
#display(HTML('./figures/figure_1.html'))

In [ ]:
display(HTML('./figures/figure_1.html'))

In [ ]:
freq_doi = df_complete.groupby(["year_d", "has_doi"]).agg(nb =("id", "size")).sort_values("year_d").reset_index()
freq_total = df_complete.groupby(["year_d"]).agg(total_by_year =("id", "size")).sort_values("year_d").reset_index()
freq_doi = freq_doi.merge(freq_total, on = ["year_d"], how ="left")
freq_doi["prop"] = freq_doi["nb"]/freq_doi["total_by_year"]



 
fig = px.area(freq_doi.sort_values(["year_d", "has_doi"], ascending=[True, False]).reset_index(), x= "year_d", y = "prop", color = "has_doi",
             title = "Evolution du taux de publications avec un DOI crossref observées en 2024", labels = {"year_d":"Années de publication", "prop": "Proportions"})
config = {'displayModeBar':True}
fig.show()
#plotly.offline.plot(fig, filename = './figures/figure_taux_publi_doi.html', config = config)
#display(HTML('./figures/figure_taux_publi_doi.html')

In [ ]:
display(HTML('./figures/figure_taux_publi_doi.html'))

### Taux de publications ouvertes de Paris 8 par année d'observation (visualisation du BSO)

<iframe id="publi.general.dynamique-ouverture.chart-taux-ouverture" title="Université Paris 8 : Taux d'accès ouvert des publications scientifiques de l'université Paris 8, avec un DOI Crossref, parues durant l'année précédente par année d'observation" width="800" height="1000" src="https://barometredelascienceouverte.esr.gouv.fr/integration/fr/publi.general.dynamique-ouverture.chart-taux-ouverture?bsoLocalAffiliation=199318270&displayComment=true&displayTitle=false&displayFooter=true&endYear=2023&lastObservationYear=2024&startYear=2013&firstObservationYear=2018&useHalId=false"></iframe>


On notera que le graphique produit par les équipes du BSO indique un taux d'ouverture de **65,7**% concernant les publications de Paris 8 parues en 2023.

### Le taux national de publications ouvertes (visualisation du BSO)

<iframe id="publi.general.dynamique-ouverture.chart-taux-ouverture" title="Taux d'accès ouvert des publications scientifiques françaises, avec un DOI Crossref, parues durant l'année précédente par année d'observation" width="800" height="1000" src="https://barometredelascienceouverte.esr.gouv.fr/integration/fr/publi.general.dynamique-ouverture.chart-taux-ouverture"></iframe>

In [ ]:
df_2023 = df_complete.loc[(df_complete.year_d<2024)]
freq_oa = df_2023.groupby(["year_d", "is_oa"]).agg(nb = ('id', 'size')).sort_values("year_d").reset_index()
freq_total = df_2023.groupby(["year_d"]).agg(total_by_year = ('id', 'size')).sort_values("year_d").reset_index()
freq_oa = freq_oa.merge(freq_total, on = ["year_d"], how ="left")
freq_oa["prop"] = freq_oa["nb"]/freq_oa["total_by_year"]


fig = px.area(freq_oa.sort_values(["year_d", "is_oa"], ascending=[True, False]).reset_index(), x= "year_d", y = "prop", color = "is_oa",
             title = "Evolution du taux d'ouverture avec ou sans DOI crossref observées en 2024", labels = {"year_d":"Années de publication", "prop": "Proportions"})
config = {'displayModeBar':True}
plotly.offline.plot(fig, filename = './figures/figure_taux_publi_ouvert.html', config = config)
#display(HTML('./figures/figure_taux_publi_ouvert.html')

In [ ]:
display(HTML('./figures/figure_taux_publi_ouvert.html'))

In [ ]:
df_2023 = df_complete.loc[(df_complete.year_d<2024) & (df_complete.has_doi==True)]
freq_oa = df_2023.groupby(["year_d", "is_oa"]).agg(nb = ('id', 'size')).sort_values("year_d").reset_index()
freq_total = df_2023.groupby(["year_d"]).agg(total_by_year = ('id', 'size')).sort_values("year_d").reset_index()
freq_oa = freq_oa.merge(freq_total, on = ["year_d"], how ="left")
freq_oa["prop"] = freq_oa["nb"]/freq_oa["total_by_year"]

fig = px.area(freq_oa.sort_values(["year_d", "is_oa"], ascending=[True, False]).reset_index(), x= "year_d", y = "prop", color = "is_oa",
             title = "Evolution du taux d'ouverture des publications avec DOI crossref observée en 2024", labels = {"year_d":"Années de publication", "prop": "Proportions"})
plotly.offline.plot(fig, filename = './figures/figure_taux_publi_avec_doi_ouvert.html', config = config)

In [ ]:
display(HTML('./figures/figure_taux_publi_avec_doi_ouvert.html'))

In [ ]:
df_2023 = df_complete.loc[(df_complete.year_d<2024) & (df_complete.is_oa==True)]
freq_oa = df_2023.groupby(["year_d", "type_id"]).agg(nb = ('id', 'size')).sort_values("year_d").reset_index()
freq_total = df_2023.groupby(["year_d"]).agg(total_by_year = ('id', 'size')).sort_values("year_d").reset_index()
freq_oa = freq_oa.merge(freq_total, on = ["year_d"], how ="left")
freq_oa["prop"] = freq_oa["nb"]/freq_oa["total_by_year"]

fig = px.area(freq_oa.sort_values(["year_d", "type_id"], ascending=[True, False]).reset_index(), x= "year_d", y = "prop", color = "type_id",
             title = "Proportion des types d'identifiants des publications en accès ouvert observée en 2024", labels = {"year_d":"Années de publication", "prop": "Proportions"})
fig.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))

df_oa = df_complete.loc[(df_complete.year_d<2024) & (df_complete.is_oa ==True)]

df_grouped = df_oa.groupby(["oa_host_type"]).agg(nb =("id", "size")).reset_index()

ax = squarify.plot(sizes=df_grouped['nb'], label=df_grouped["oa_host_type"], value=df_grouped['nb'], alpha=.8, color = ["sandybrown","gold","green","skyblue","lightgrey"] )
ax.set_axis_off()
ax.set_title("Nombre de publications ouvertes observées en 2024 par type d'accès")

glue("squarify_oa", fig)


### Accès ouvert et voies d'accès

Toutefois, ce "taux d'ouverture" (qui correspond à la variable `is_oa`) ne prend pas en compte les différents types d'accès. On distingue en effet 6 "voies" d'ouverture, désignées à l'aide de couleurs :

1. la {term}`voie diamant` : les articles sont publiés dans une revue scientifique ouverte ne demandant ni de frais de publication ni de frais d'abonnement. C'est le cas par exemple de la revue *Journal of Theoretical, Computational and Applied Mechanics* citée plus haut.

2. la {term}`voie dorée` : Publication en accès libre, financée par des frais de publication (APC - Article Processing Charges) payés par les auteurs ou leurs institutions. 

3. la {term}`voie Verte` : Dépôt d’articles dans des archives ouvertes après un éventuel embargo imposé par l’éditeur. Souvent, le document déposé correspond à la dernière version avant publication.

4. la {term}`voie hybride` : Une partie des publications d'une revue sont en accès libre, soit parce que les auteurs ont payé des frais de publication, soit parce les articles ont acquis une certaine "ancienneté".

5. la {term}`voie platine` : Journal dont la publication en open access ne nécessite pas de frais de publication. Le coût associé à la publication est couvert par d'autres moyens : bénévolat, dons, prix, etc. (traduit de ["Open Access colours: green, gold, diamond, hybrid and more"](https://researchtips.ugent.be/en/tips/00000461/#Platinum))

6. la {term}`voie bronze` : la publication est mise en accès libre sur le site de l'éditeur, mais sans licence ouverte explicite.

Dans les données fournies par l'équipe du BSO, le type d'accès est précisé pour chaque publication à chaque date d'observation à partir des informations d'Unpaywall. Le tableau ci-dessous illustre la relation entre "accès ouvert" (`is_oa`) et les différentes voies d'accès (`unpaywall_oa_status`). On notera que 654 publications en accès ouvert ne sont pas référencées sur Unpaywall puisqu'elles n'ont pas de DOI et que 40 sont indéfinies.

In [ ]:
glue('table_oa_status', oa_status1.fillna("").style.set_caption("Ouverture et voies d'accès"))

20 % des publications avec un DOI observées en 2024 étaient ouverte via la "voie dorée". Elles représentent un tiers des publications de Paris 8 en accès ouvert.

```{glue:figure} squarify_oa
:figwidth: 600px
:name: "squarify-oa"


```

In [ ]:
fig, axes = plt.subplots(2, figsize=(10,10))



df_2023 = df_complete.loc[(df_complete.year_d<2024) & (df_complete.has_doi==True)]
freq_oa = df_2023.groupby(["year_d", "is_oa"]).agg(nb = ('id', 'size')).sort_values("year_d").reset_index()
freq_total = df_2023.groupby(["year_d"]).agg(total_by_year = ('id', 'size')).sort_values("year_d").reset_index()
freq_oa = freq_oa.merge(freq_total, on = ["year_d"], how ="left")
freq_oa["prop"] = freq_oa["nb"]/freq_oa["total_by_year"]


# set seaborn style
#sns.set_theme()
 
# Data
x=[x for x in freq_oa.year_d.unique()]
y=[]
for n, z in enumerate(freq_oa.is_oa.unique()):
    v = [v for v in freq_oa.prop.loc[freq_oa.is_oa == z]]
    y.append(v)

# Plot
axes[0].stackplot(x,y, labels=['Fermé','Ouvert'])
axes[0].set_xlabel("Années")
axes[0].set_ylabel("Proportion")
axes[0].legend(loc='upper left')
axes[0].set_title("Evolution du taux d'ouverture des publications avec un DOI crossref observée en 2024")



df_2023 = df_complete.loc[(df_complete.year_d<2024)]
freq_oa = df_2023.groupby(["year_d", "is_oa"]).agg(nb = ('id', 'size')).sort_values("year_d").reset_index()
freq_total = df_2023.groupby(["year_d"]).agg(total_by_year = ('id', 'size')).sort_values("year_d").reset_index()
freq_oa = freq_oa.merge(freq_total, on = ["year_d"], how ="left")
freq_oa["prop"] = freq_oa["nb"]/freq_oa["total_by_year"]


# set seaborn style
#sns.set_theme()
 
# Data
x=[x for x in freq_oa.year_d.unique()]
y=[]
for n, z in enumerate(freq_oa.is_oa.unique()):
    v = [v for v in freq_oa.prop.loc[freq_oa.is_oa == z]]
    y.append(v)

# Plot
axes[1].stackplot(x,y, labels=['Fermé','Ouvert'])
axes[1].set_xlabel("Années")
axes[1].set_ylabel("Proportion")
axes[1].legend(loc='upper left')
axes[1].set_title("Evolution du taux d'ouverture des publications avec ou sans DOI crossref observée en 2024")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, figsize=(10,5))
df_oa = df_complete.loc[(df_complete.year_d<2024) & (df_complete.is_oa==True) & (df_complete.has_doi==True) &(df_complete.oa_host_type!="unknown")]

df_grouped = df_oa.groupby(["year_d","oa_host_type"]).agg(nb =("id", "size")).sort_values(["year_d","nb"], ascending=[True, False]).reset_index()

sns.barplot(data=df_grouped, x="year_d", y="nb", hue="oa_host_type", dodge=False);

axes.set_xlabel("Années")
axes.set_ylabel("Nombre de publications")
axes.set_title("Distribution annuelle des publications ouvertes avec DOI Crossref selon le type d'accès (observation 2024)");


In [ ]:
fig = px.bar(df_grouped, x="year_d", y="nb", color="oa_host_type" , title = "Distribution annuelle des publications ouvertes avec DOI Crossref selon le type d'accès (observation 2024)",
            labels = {"year_d":"Années de publication", "nb": "Nombre de publications"})
plotly.offline.plot(fig, filename = './figures/figure_nombre_publi_avec_doi_type_acces.html', config = config)

In [ ]:
display(HTML('./figures/figure_nombre_publi_avec_doi_type_acces.html'))

In [ ]:
fig, axes = plt.subplots(1, figsize=(10,5))
df_oa = df_complete.loc[(df_complete.year_d<2024) & (df_complete.has_doi==True) & (df_complete.unpaywall_oa_status_rec.isin(["gold","green","hybrid","bronze"]))]

df_grouped = df_oa.groupby(["year_d","unpaywall_oa_status_rec"]).agg(nb =("id", "size")).sort_values("year_d").reset_index()

sns.barplot(data=df_grouped, x="year_d", y="nb", hue="unpaywall_oa_status_rec", dodge=False);

axes.set_xlabel("Années")
axes.set_ylabel("Nombre de publications")
axes.set_title("Distribution annuelle des publications ouvertes selon la voie d'accès (observation 2024)");





In [ ]:
check_id = []
df0.loc[df0.is_oa==True, 'is_oa_rec']=1
df0.loc[df0.is_oa==False, 'is_oa_rec']=0
for n, x in enumerate(df0.id.unique()):
    dftmp = df0.loc[df0.id == x]
    somme_oa = np.sum(dftmp.is_oa_rec)
    if somme_oa > 0 and somme_oa < len(dftmp):
        check_id.append(x)
    else:
        pass
        